# 멀티 모달 기본
이미지를 읽어서 LLM이 설명

In [1]:
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-core
!pip install python-dotenv
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.43.0
    Uninstalling openai-2.43.0:
      Successfully uninstalled openai-2.43.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4.8:
      Successfully uninstalled langchain-core-1.4.8
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.

In [3]:
import base64
from pathlib import Path
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

import os
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY가 없어요")

llm = ChatOpenAI(model="gpt-4.1-mini", temperature="0.5")

In [8]:
# Image를 base64 data url 형식으로 변환하는 함수 생성
def encode_image_to_data_url(img_path:str)-> str:
  ext = Path(img_path).suffix.lower().replace(".","") # 이미지 확장자 추출(.이전은 전부 없애고 확장자만 추출됨)
  # print(f"img_path:{img_path} ⇒ ext:{ext}")

  # OpenAI vision에서는 jpg보다 jpeg Mime Type를 선호
  if ext == "jpg": ext = 'jpeg'

  with open(img_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")
    # print(b64)

  return f'data:image/{ext};base64,{b64}' # LangChain MultiModal에서 요구하는 data URL 형식으로 반환

# 이미지를 1장 보내면 LLM이 설명을 달아주는 함수 생성
def discribe_image(img_path:str)-> str:
  img_url = encode_image_to_data_url(img_path)

  # HumanMessage 형식으로 메세지 작성
  msg = HumanMessage(
      content=[
          # LLM이 수행할 요청
        {
          "type": "text",
          "text": "이 이미지의 내용을 상세하게 설명해줘"
        },
          # 이미지 제공 - 여기에서 type이 img_url이여서 encode_image_to_data_url가 필요함.
        {
          "type": "image_url",
          "image_url": {"url": img_url}
        },
      ]
  )
  res = llm.invoke([msg])
  return res.content

In [9]:
if __name__ == "__main__":
  img_path = "person.jpeg"

  print("이미지 설명 결과 : ")
  print(discribe_image(img_path))

이미지 설명 결과 : 
이 이미지는 다섯 명의 여성 멤버들이 함께 있는 사진입니다. 다섯 명 모두 하늘색 또는 흰색 계열의 블라우스를 입고 있으며, 블라우스는 프릴 디테일이 있는 디자인입니다. 각 멤버들은 머리에 분홍빛의 큰 곰돌이 귀 모양의 헤어밴드를 착용하고 있습니다. 멤버들은 소파와 쿠션이 놓인 아늑한 실내 공간에 앉거나 기대어 있는 자세를 취하고 있습니다.

이미지 왼쪽 상단에는 하얀 실로 짠 듯한 느낌의 독특한 글씨체로 어떤 문자가 쓰여 있고, 오른쪽 하단에는 "Let's glow up!"이라는 문구와 함께 "RESCENE 2nd MINI ALBUM Glow Up"이라는 텍스트가 적힌 라벨이 붙어 있습니다. 이로 보아 이 이미지는 'RESCENE'이라는 그룹의 두 번째 미니 앨범 'Glow Up'의 앨범 커버 혹은 프로모션 이미지로 추정됩니다.

전체적으로 부드럽고 몽환적인 분위기와 함께 멤버들의 청순하고 세련된 이미지를 강조한 사진입니다.
